In [1]:
!pip install pandas numpy

import pandas as pd
import numpy as np

In [4]:
from google.colab import files
uploaded = files.upload()



Saving Question 4 - Task.csv to Question 4 - Task (1).csv


In [5]:
df = pd.read_csv(list(uploaded.keys())[0])
df.head(2)

,segment_url_link,Human,Model H,Model i,Model k,Model l,Model m,Model n,Unnamed: 8
0,https://storage.googleapis.com/testing_audio_f...,वही अपना खेती बाड़ी और क्या,वही अपना खेती बाड़ी और क्या,वही अपना खेती बाड़ी और क्या,वही अपना खेती बाड़ी और क्या?,वही अपना खेती बाड़ी और क्या,वही अपना खेतीबाड़ी और क्या,वही अपना खेती बाड़ी और क्या,NaN
1,https://storage.googleapis.com/testing_audio_f...,मौनता का अर्थ क्या होता है,मौनता का अर्थ क्या होता है,मौनता का अर्थ क्या होता है?,मौन तागार थके होतई।,मोनता का अर्थ है क्या होता है,मोन ताका हर थक्या होताहए,मौनता का हर थका होता है,NaN


In [6]:
def tokenize(text):
    return str(text).strip().split()

In [8]:
def align_sequences(seqs):
    max_len = max(len(seq) for seq in seqs)

    aligned = []

    for seq in seqs:
        padded = seq + ["<pad>"] * (max_len - len(seq))
        aligned.append(padded)

    return list(zip(*aligned))

In [9]:
def build_lattice(models, reference):
    tokenized = [tokenize(m) for m in models]
    tokenized.append(tokenize(reference))

    aligned = align_sequences(tokenized)

    lattice = []

    for position in aligned:
        variants = set([w for w in position if w != "<pad>"])
        lattice.append(list(variants))

    return lattice

In [11]:
def compute_wer(ref, hyp):
    ref = tokenize(ref)
    hyp = tokenize(hyp)

    dp = np.zeros((len(ref)+1, len(hyp)+1))

    for i in range(len(ref)+1):
        dp[i][0] = i
    for j in range(len(hyp)+1):
        dp[0][j] = j

    for i in range(1, len(ref)+1):
        for j in range(1, len(hyp)+1):
            if ref[i-1] == hyp[j-1]:
                dp[i][j] = dp[i-1][j-1]
            else:
                dp[i][j] = 1 + min(
                    dp[i-1][j],
                    dp[i][j-1],
                    dp[i-1][j-1]
                )

    return dp[len(ref)][len(hyp)] / len(ref)

In [12]:
def compute_lattice_wer(pred, lattice):
    pred_tokens = tokenize(pred)

    errors = 0

    for i in range(len(lattice)):
        if i >= len(pred_tokens) or pred_tokens[i] not in lattice[i]:
            errors += 1

    return errors / len(lattice)

In [15]:
row = df.iloc[0]

reference = row["Human"]

models = [
    row["Model H"],
    row["Model i"],
    row["Model k"],
    row["Model l"],
    row["Model m"],
    row["Model n"]
]

In [16]:
lattice = build_lattice(models, reference)

for i, bin in enumerate(lattice):
    print(f"Position {i+1}: {bin}")

Position 1: ['वही']
Position 2: ['अपना']
Position 3: ['खेतीबाड़ी', 'खेती']
Position 4: ['बाड़ी', 'बाड़ी', 'और']
Position 5: ['और', 'क्या']
Position 6: ['क्या?', 'क्या']


In [18]:
results = []

for i, model_output in enumerate(models):

    normal = compute_wer(reference, model_output)
    lattice_wer = compute_lattice_wer(model_output, lattice)

    results.append([
        f"Model {i+1}",
        normal,
        lattice_wer
    ])

results_df = pd.DataFrame(results, columns=[
    "Model", "Normal WER", "Lattice WER"
])

results_df

,Model,Normal WER,Lattice WER
0,Model 1,0.000000,0.000000
1,Model 2,0.166667,0.000000
2,Model 3,0.166667,0.000000
3,Model 4,0.000000,0.000000
4,Model 5,0.333333,0.166667
5,Model 6,0.000000,0.000000
